# Disaster Tweets Prediction


In [2]:
!wget https://raw.githubusercontent.com/umerkang66/ai-ml-dl/refs/heads/master/tensorflow-bootcamp/03-computer-vision-tf/helper_functions.py

--2026-08-22 17:28:40--  https://raw.githubusercontent.com/umerkang66/ai-ml-dl/refs/heads/master/tensorflow-bootcamp/03-computer-vision-tf/helper_functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10836 (11K) [text/plain]
Saving to: ‘helper_functions.py.1’

helper_functions.py 100%[===================>]  10.58K  --.-KB/s    in 0s      

2026-08-22 17:28:40 (122 MB/s) - ‘helper_functions.py.1’ saved [10836/10836]



In [3]:
from helper_functions import (
    unzip_data,
    plot_loss_curves,
    compare_historys,
    make_confusion_matrix,
)

## Kaggle Intro to NLP Dataset


In [4]:
!wget https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip

--2026-08-22 17:28:47--  https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 34.153.2.27, 34.3.1.27, 34.153.3.27, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|34.153.2.27|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 607343 (593K) [application/zip]
Saving to: ‘nlp_getting_started.zip.1’

nlp_getting_started 100%[===================>] 593.11K  1.28MB/s    in 0.5s    

2026-08-22 17:28:48 (1.28 MB/s) - ‘nlp_getting_started.zip.1’ saved [607343/607343]



In [5]:
unzip_data("nlp_getting_started.zip")

## Read the Data


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split

train_df = pd.read_csv("train.csv")[["text", "target"]]

train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42)

test_df = pd.read_csv("test.csv")[["text"]]

In [8]:
train_df.head()

,text,target
4620,'McFadden Reportedly to Test Hamstring Thursda...,0
2858,w--=-=-=-[ NEMA warns Nigerians to prepare for...,1
3098,When I was cooking earlier I got electrocuted ...,0
3751,I'm On Fire. http://t.co/WATsmxYTVa,0
5285,More than 40 families affected by the fatal ou...,1


## Shuffle the training data


In [9]:
train_df_shuffled = train_df.sample(frac=1, random_state=42)

In [10]:
train_df_shuffled.head()

,text,target
3347,78 passengers evacuated safely after Green Lin...,1
6177,@KIRO7Seattle Just saw a Bomb Squad car headin...,1
6970,@Kamunt Holy crap it's been forever since I sa...,0
1497,Learning from the Legacy of a Catastrophic Eru...,1
3453,luke + microphone = exploded ovaries,0


In [11]:
val_df.head()

,text,target
2644,So you have a new weapon that can cause un-ima...,1
2227,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,Aftershock back to school kick off was great. ...,0
6845,in response to trauma Children of Addicts deve...,0


In [12]:
test_df.head()

,text
0,Just happened a terrible car crash
1,"Heard about #earthquake is different cities, s..."
2,"there is a forest fire at spot pond, geese are..."
3,Apocalypse lighting. #Spokane #wildfires
4,Typhoon Soudelor kills 28 in China and Taiwan


In [13]:
train_df.target.value_counts()

,count
target,
0,3916
1,2935


In [14]:
len(train_df), len(test_df)

(6851, 3263)

## Let's visualize some random samples


In [15]:
import random

random_index = random.randint(0, len(train_df) - 1)

for _, row in train_df.iloc[random_index : random_index + 5].iterrows():
    text, target = row

    print(
        f"Target: {target}", "(real disaster)" if target > 0 else "(not real disaster)"
    )
    print(f"Text: {text}\n")

Target: 0 (not real disaster)
Text: Why did God order obliteration of ancient Canaanites? http://t.co/pKKcdWjyg0 via @worldnetdaily

Target: 0 (not real disaster)
Text: @CW_Hoops you better make all your shots tomorrow cause I'm recording and flames will be thrown tomorrow

Target: 1 (real disaster)
Text: @DougMartin17 Fireman Ed runs into burning buildings while others are running out Doug he deserves your respect??????

Target: 0 (not real disaster)
Text: @LongBreastYat Yeah I don't think he's elite either I think Hazard is the better player too. But not by much

Target: 1 (real disaster)
Text: The Catastrophic Effects of Hiroshima and Nagasaki Atomic Bombings Still Being Felt Today http://t.co/rNqEBAyCVM



## Converting text into numbers


In [16]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

In [17]:
average_max_length = round(
    sum([len(i.split()) for i in train_df["text"]]) / len(train_df)
)

In [18]:
# we have already defaulted this, but here we are setting again
max_length = average_max_length  # how many words from a tweet our model will see


text_vectorizer = TextVectorization(
    max_tokens=10000,  # None = no limit on the number of tokens vocab can have
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    ngrams=None,  # treat each word as a token
    output_mode="int",  # convert tokens into integers
    output_sequence_length=max_length,  # pad all outputs to be max_length tokens long
    pad_to_max_tokens=True,
)

In [19]:
# fit the text vectorizer to the training text

text_vectorizer.adapt(train_df["text"])

In [20]:
# create a sample text and pass it through our text vectorizer instance

sample_text = "There's a flood in my street!"

text_vectorizer([sample_text])

<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[282,   3, 206,   4,  13, 674,   0,   0,   0,   0,   0,   0,   0,
          0,   0]])>

In [21]:
# choose a random text from the training dataset and pass it through the text vectorizer

import random

random_sentence = random.choice(train_df["text"])

print(f"Original text:\n{random_sentence} \n")
print(f"Vectorized version:\n{text_vectorizer([random_sentence])}")

Original text:
What tropical storm? #guillermo by hawaiianpaddlesports http://t.co/LgPgAjgomY http://t.co/FKd1mBTB68 

Vectorized version:
[[  54 1947   98 2102   18    1    1    1    0    0    0    0    0    0
     0]]


In [22]:
# get the unique words in the vocabulary of our text vectorizer instance

words_in_vocab = text_vectorizer.get_vocabulary()

print("Top 5 words in vocab: ", words_in_vocab[:5])
print("Bottom 5 words in vocab: ", words_in_vocab[-5:])
print("Number of words in vocab: ", len(words_in_vocab))

Top 5 words in vocab:  ['', '[UNK]', np.str_('the'), np.str_('a'), np.str_('in')]
Bottom 5 words in vocab:  [np.str_('pakthey'), np.str_('pakistan\x89Ûªs'), np.str_('pakistans'), np.str_('pajamas'), np.str_('paints')]
Number of words in vocab:  10000


In [23]:
from tensorflow.keras.layers import Embedding

embedding = Embedding(
    input_dim=len(
        words_in_vocab
    ),  # total vocabulary size (i.e. number of unique tokens in the text)
    output_dim=128,  # set size of embedding vector
    input_length=max_length,  # how long is each input
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [24]:
random_sentence = random.choice(train_df["text"])


print(f"Original text:\n{random_sentence} \n")
print(f"Vectorized version:\n{text_vectorizer([random_sentence])}")

sample_embed = embedding(text_vectorizer([random_sentence]))


print(f"Embedded version:\n{sample_embed}")
print(f"Embedded version shape:\n{sample_embed.shape}")

Original text:
Trafford Centre film fans angry after Odeon cinema evacuated following false fire alarm http://t.co/pFMn63VnAm http://t.co/vKwqbOJFJc 

Vectorized version:
[[2244  818  906 1078  823   38 2362 2152  602  949 1408   44 1031    1
     1]]
Embedded version:
[[[ 0.01173438 -0.00707211  0.01141305 ... -0.01643046  0.01749349
   -0.0281433 ]
  [ 0.02698136 -0.04004193  0.00162643 ... -0.04347695  0.04817793
    0.03687229]
  [-0.03065894 -0.03281289  0.0026155  ...  0.04822308 -0.03997706
   -0.01715888]
  ...
  [ 0.00749006  0.01238929 -0.03541251 ...  0.0176958   0.0179845
    0.00086801]
  [ 0.01340063 -0.0173291  -0.01540854 ... -0.04156197 -0.0029039
   -0.01139442]
  [ 0.01340063 -0.0173291  -0.01540854 ... -0.04156197 -0.0029039
   -0.01139442]]]
Embedded version shape:
(1, 15, 128)


In [25]:
sample_embed[0][0], sample_embed[0][0].shape

(<tf.Tensor: shape=(128,), dtype=float32, numpy=
 array([ 1.1734378e-02, -7.0721135e-03,  1.1413049e-02,  1.9626189e-02,
        -2.9783918e-02,  4.3426383e-02, -4.3473471e-02,  7.2756521e-03,
        -4.9590956e-02,  2.3190226e-02,  1.3749067e-02, -3.6555301e-02,
        -3.4660589e-02,  4.7221910e-02,  3.8037483e-02, -4.9936067e-02,
         2.8818678e-02, -1.4314175e-02,  2.4507727e-02, -1.2219332e-02,
        -4.2154681e-02, -2.4088239e-02,  2.2675063e-02,  5.5744424e-03,
        -1.0993637e-02, -5.8489814e-03, -3.3416569e-02, -4.3787576e-02,
        -4.7414161e-02,  2.2505891e-02, -4.7483921e-02,  4.0501188e-02,
         2.5749650e-02, -3.8052477e-02,  2.2517350e-02,  3.8725328e-02,
         3.5696737e-03,  4.4431631e-02,  4.0353645e-02,  3.0257311e-02,
        -2.2979094e-02,  1.4240887e-02,  2.2455905e-02, -3.9021395e-02,
        -9.5583498e-05,  2.5788333e-02,  5.3635724e-03,  3.4489743e-03,
         2.3596022e-02, -4.5119151e-03,  3.2434475e-02,  3.1329241e-02,
         4.1349

## Experiment Models

- Model 0: Naive Bayes (Baseline)
- Model 1: Feed Forward Neural Network (dense model)
- Model 2: LSTM model (RNN)
- Model 3: GRU model (RNN)
- Model 4: Bidirectional-LSTM (RNN)
- Model 5: 1D Convolutional Neural Network (CNN)
- Model 6: TensorFlow Hub Pretrained Feature Extractor (Universal Sentence Encoder - Transfer Learning)
- Model 7: TensorFlow Hub Pretrained Feature Extractor (10% of training data)


## MODEL_0: BaseLine Naive Bayes


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Create tokenization and modelling pipeline
model_0 = Pipeline(
    [
        ("tfidf", TfidfVectorizer()),  # convert words to numbers using tfidf
        ("clf", MultinomialNB()),  # model the text
    ]
)

# Fit the pipeline to the training data
model_0.fit(train_df["text"], train_df["target"])

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])

In [27]:
# Evaluate baseline model
baseline_score = model_0.score(val_df["text"], val_df["target"])
print(f"Our baseline model achieves an accuracy of: {baseline_score * 100:.2f}%")

Our baseline model achieves an accuracy of: 77.82%


In [28]:
# Make predictions
baseline_preds = model_0.predict(val_df["text"])

In [29]:
# Calculate baseline results
from helper_functions import calculate_results

baseline_results = calculate_results(y_true=val_df["target"], y_pred=baseline_preds)
baseline_results

{'accuracy': 77.82152230971128,
 'precision': 0.792992256322435,
 'recall': 0.7782152230971129,
 'f1': 0.7703527809038112}

## MODEL_1: FFNN


In [30]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D

In [31]:
inputs = tf.keras.layers.Input(
    shape=(1,), dtype=tf.string
)  # inputs are 1 dimenional strings (i.e. sentences)

x = text_vectorizer(inputs)  # turn the input text into numbers
x = embedding(x)  # create an embedding of the numberized inputs

x = GlobalAveragePooling1D()(x)

outputs = Dense(1, activation="sigmoid")(x)


model_1 = tf.keras.Model(inputs, outputs, name="model_1_dense")

In [32]:
model_1.summary()

Model: "model_1_dense"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization              │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,280,129 (4.88 MB)

 Trainable params: 1,280,129 (4.88 MB)

 Non-trainable params: 0 (0.00 B)

In [33]:
model_1.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

model_1_history = model_1.fit(
    train_df["text"].to_numpy(),
    train_df["target"].to_numpy(),
    epochs=5,
    validation_data=(val_df["text"].to_numpy(), val_df["target"].to_numpy()),
)

Epoch 1/5


215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6998 - loss: 0.6070 - val_accuracy: 0.7612 - val_loss: 0.5422
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8162 - loss: 0.4413 - val_accuracy: 0.7861 - val_loss: 0.4854
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8624 - loss: 0.3484 - val_accuracy: 0.7822 - val_loss: 0.4760
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8897 - loss: 0.2847 - val_accuracy: 0.7900 - val_loss: 0.4884
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9123 - loss: 0.2382 - val_accuracy: 0.7861 - val_loss: 0.5035


In [34]:
model1_pred_probs = model_1.predict(val_df["text"].to_numpy())

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


In [35]:
model1_preds = tf.squeeze(tf.round(model1_pred_probs))

model1_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 0., 0., 0., 1., 0., 0., 0., 0., 1.], dtype=float32)>

In [36]:
model_1_results = calculate_results(y_true=val_df["target"], y_pred=model1_preds)

model_1_results

{'accuracy': 78.60892388451444,
 'precision': 0.7882539924658478,
 'recall': 0.7860892388451444,
 'f1': 0.7831951936309017}

## MODEL2: LSTM


In [37]:
from tensorflow.keras.layers import LSTM

inputs = tf.keras.layers.Input(shape=(1,), dtype=tf.string)

x = text_vectorizer(inputs)
x = embedding(x)
x = LSTM(64, return_sequences=True)(x)
x = LSTM(64)(x)
outputs = Dense(1, activation="sigmoid")(x)

model_2 = tf.keras.Model(inputs, outputs, name="model_2_LSTM")

model_2.summary()

Model: "model_2_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization              │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 15, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,362,497 (5.20 MB)

 Trainable params: 1,362,497 (5.20 MB)

 Non-trainable params: 0 (0.00 B)

In [38]:
# Compile model
model_2.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

# Fit model
model_2_history = model_2.fit(
    train_df["text"].to_numpy(),
    train_df["target"].to_numpy(),
    epochs=5,
    validation_data=(val_df["text"].to_numpy(), val_df["target"].to_numpy()),
)

Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9206 - loss: 0.2211 - val_accuracy: 0.7795 - val_loss: 0.5713
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9422 - loss: 0.1600 - val_accuracy: 0.7651 - val_loss: 0.6052
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9504 - loss: 0.1301 - val_accuracy: 0.7572 - val_loss: 0.7832
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9588 - loss: 0.1070 - val_accuracy: 0.7480 - val_loss: 0.9589
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9657 - loss: 0.0826 - val_accuracy: 0.7546 - val_loss: 0.9618


In [39]:
# Make predictions on the validation set
model_2_pred_probs = model_2.predict(val_df["text"].to_numpy())

# Convert prediction probabilities to 0 or 1 binary classes
model_2_preds = tf.squeeze(tf.round(model_2_pred_probs))

# Calculate evaluation metrics (Accuracy, Precision,Recall, F1-score)
from helper_functions import calculate_results

model_2_results = calculate_results(
    y_true=val_df["target"].to_numpy(), y_pred=model_2_preds
)
model_2_results

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


{'accuracy': 75.45931758530183,
 'precision': 0.7537853551898928,
 'recall': 0.7545931758530183,
 'f1': 0.7538089191165337}

## MODEL3: GRU


In [40]:
from tensorflow.keras.layers import GRU

inputs = tf.keras.layers.Input(shape=(1,), dtype=tf.string)

x = text_vectorizer(inputs)
x = embedding(x)
x = GRU(64, return_sequences=True)(x)
x = GRU(64)(x)
outputs = Dense(1, activation="sigmoid")(x)

model_3 = tf.keras.Model(inputs, outputs, name="model_3_GRU")

model_3.summary()

Model: "model_3_GRU"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization              │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 15, 64)         │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,342,273 (5.12 MB)

 Trainable params: 1,342,273 (5.12 MB)

 Non-trainable params: 0 (0.00 B)

In [41]:
# Compile model
model_3.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

# Fit model
model_3_history = model_3.fit(
    train_df["text"].to_numpy(),
    train_df["target"].to_numpy(),
    epochs=5,
    validation_data=(val_df["text"].to_numpy(), val_df["target"].to_numpy()),
)

Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9437 - loss: 0.1405 - val_accuracy: 0.7703 - val_loss: 0.6713
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9691 - loss: 0.0858 - val_accuracy: 0.7625 - val_loss: 0.8611
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9721 - loss: 0.0669 - val_accuracy: 0.7625 - val_loss: 1.2347
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9753 - loss: 0.0593 - val_accuracy: 0.7717 - val_loss: 1.0401
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9753 - loss: 0.0597 - val_accuracy: 0.7415 - val_loss: 1.3817


In [42]:
# Make predictions on the validation set
model_3_pred_probs = model_3.predict(val_df["text"].to_numpy())

# Convert prediction probabilities to 0 or 1 binary classes
model_3_preds = tf.squeeze(tf.round(model_3_pred_probs))

# Calculate evaluation metrics (Accuracy, Precision,Recall, F1-score)
from helper_functions import calculate_results

model_3_results = calculate_results(
    y_true=val_df["target"].to_numpy(), y_pred=model_3_preds
)
model_3_results

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


{'accuracy': 74.1469816272966,
 'precision': 0.740774483809133,
 'recall': 0.7414698162729659,
 'f1': 0.7409663005244673}

## MODEL4: Bidirectional LSTM


In [43]:
from tensorflow.keras.layers import Bidirectional, LSTM

inputs = tf.keras.layers.Input(shape=(1,), dtype=tf.string)

x = text_vectorizer(inputs)
x = embedding(x)
# x = Bidirectional(LSTM(64, return_sequences=True))(x) # Stacking bidirectional RNNs
x = Bidirectional(LSTM(64))(x)
outputs = Dense(1, activation="sigmoid")(x)

model_4 = tf.keras.Model(inputs, outputs, name="model_4_Bidirectional_LSTM")

model_4.summary()

Model: "model_4_Bidirectional_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization              │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,378,945 (5.26 MB)

 Trainable params: 1,378,945 (5.26 MB)

 Non-trainable params: 0 (0.00 B)

In [44]:
# Compile model
model_4.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

# Fit model
model_4_history = model_4.fit(
    train_df["text"].to_numpy(),
    train_df["target"].to_numpy(),
    epochs=5,
    validation_data=(val_df["text"].to_numpy(), val_df["target"].to_numpy()),
)

Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9620 - loss: 0.1089 - val_accuracy: 0.7493 - val_loss: 0.9705
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9768 - loss: 0.0513 - val_accuracy: 0.7441 - val_loss: 1.3135
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9774 - loss: 0.0462 - val_accuracy: 0.7362 - val_loss: 1.5531
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9806 - loss: 0.0400 - val_accuracy: 0.7559 - val_loss: 1.4100
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9809 - loss: 0.0411 - val_accuracy: 0.7572 - val_loss: 1.6811


In [45]:
# Make predictions on the validation set
model_4_pred_probs = model_4.predict(val_df["text"].to_numpy())

# Convert prediction probabilities to 0 or 1 binary classes
model_4_preds = tf.squeeze(tf.round(model_4_pred_probs))

# Calculate evaluation metrics (Accuracy, Precision,Recall, F1-score)
from helper_functions import calculate_results

model_4_results = calculate_results(
    y_true=val_df["target"].to_numpy(), y_pred=model_4_preds
)
model_4_results

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


{'accuracy': 75.7217847769029,
 'precision': 0.7564500894663291,
 'recall': 0.7572178477690289,
 'f1': 0.7558650969882834}

## MODEL5: Conv1D (1D Convolutional Neural Network)


In [46]:
from tensorflow.keras.layers import Conv1D, GlobalMaxPool1D

inputs = tf.keras.layers.Input(shape=(1,), dtype=tf.string)

x = text_vectorizer(inputs)
x = embedding(x)
x = Conv1D(filters=64, kernel_size=5, strides=1, activation="relu", padding="valid")(x)
x = GlobalMaxPool1D()(x)

# 2 dense layers for classification
x = Dense(64, activation="relu")(x)  # Optional dense layer
outputs = Dense(1, activation="sigmoid")(x)

model_5 = tf.keras.Model(inputs, outputs, name="model_5_Conv1D")

model_5.summary()

Model: "model_5_Conv1D"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization              │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 11, 64)         │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,325,249 (5.06 MB)

 Trainable params: 1,325,249 (5.06 MB)

 Non-trainable params: 0 (0.00 B)

In [47]:
# Compile model
model_5.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

# Fit model
model_5_history = model_5.fit(
    train_df["text"].to_numpy(),
    train_df["target"].to_numpy(),
    epochs=5,
    validation_data=(val_df["text"].to_numpy(), val_df["target"].to_numpy()),
)

Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9565 - loss: 0.1194 - val_accuracy: 0.7507 - val_loss: 1.0499
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9740 - loss: 0.0638 - val_accuracy: 0.7231 - val_loss: 1.2670
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9796 - loss: 0.0447 - val_accuracy: 0.7323 - val_loss: 1.5412
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9818 - loss: 0.0372 - val_accuracy: 0.7441 - val_loss: 1.7491
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9818 - loss: 0.0333 - val_accuracy: 0.7362 - val_loss: 1.8687


In [48]:
# Make predictions on the validation set
model_5_pred_probs = model_5.predict(val_df["text"].to_numpy())

# Convert prediction probabilities to 0 or 1 binary classes
model_5_preds = tf.squeeze(tf.round(model_5_pred_probs))

# Calculate evaluation metrics (Accuracy, Precision,Recall, F1-score)
from helper_functions import calculate_results

model_5_results = calculate_results(
    y_true=val_df["target"].to_numpy(), y_pred=model_5_preds
)
model_5_results

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


{'accuracy': 73.62204724409449,
 'precision': 0.7358458439027336,
 'recall': 0.7362204724409449,
 'f1': 0.7360012787549572}

## MODEL6: Transfer Learning (TensorFlow Hub Pretrained Universal Sentence Encoder)

Now we'll use a pretrained feature extractor model from TensorFlow Hub: [Universal Sentence Encoder (USE)](https://tfhub.dev/google/universal-sentence-encoder/4).


In [49]:
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras import layers


# Create a custom Keras 3 Layer wrapping Universal Sentence Encoder
class USEEmbeddingLayer(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        # Load the Universal Sentence Encoder module from TF Hub
        self.encoder = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")

    def call(self, inputs):
        # Flatten input to 1D strings tensor
        flat_inputs = tf.reshape(tf.cast(inputs, tf.string), [-1])
        return self.encoder(flat_inputs)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], 512)

    def get_config(self):
        return super().get_config()

In [48]:
use_layer = USEEmbeddingLayer(name="USE")
sample_sentence = "There's a flood in my street!"

sample_input = tf.constant([sample_sentence])
sample_embedded = use_layer(sample_input)

print(f"Original text:\n{sample_sentence}\n")
print(f"Embedded version (first 50 values):\n{sample_embedded[0][:50]}...\n")
print(f"Embedded vector shape: {sample_embedded.shape}")

Original text:
There's a flood in my street!

Embedded version (first 50 values):
[-0.01157027  0.02485911  0.02878049 -0.01271501  0.0397154   0.08827761
  0.02680985  0.05589838 -0.01068731 -0.0059729   0.00639323 -0.01819521
  0.00030817  0.09105889  0.05874642 -0.03180628  0.01512472 -0.05162928
  0.00991366 -0.06865346 -0.04209308  0.0267898   0.03011007  0.00321065
 -0.00337969 -0.04787356  0.02266722 -0.00985928 -0.04063612 -0.01292093
 -0.04666385  0.05630299 -0.03949254  0.00517688  0.02495828 -0.07014439
  0.02871511  0.04947681 -0.00633974 -0.08960193  0.02807117 -0.00808362
 -0.01360602  0.05998651 -0.10361787 -0.05195372  0.00232956 -0.02332529
 -0.03758107  0.0332773 ]...

Embedded vector shape: (1, 512)


In [50]:
# Build Model 6 using Functional API with the modern Keras 3 USE layer
inputs = layers.Input(shape=(), dtype=tf.string, name="input_text")
x = USEEmbeddingLayer(name="universal_sentence_encoder")(inputs)
x = layers.Dense(64, activation="relu")(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model_6 = tf.keras.Model(inputs=inputs, outputs=outputs, name="model_6_USE")
model_6.summary()

Model: "model_6_USE"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_text (InputLayer)         │ (None)                 │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ universal_sentence_encoder      │ (None, 512)            │             0 │
│ (USEEmbeddingLayer)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │        32,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,897 (128.50 KB)

 Trainable params: 32,897 (128.50 KB)

 Non-trainable params: 0 (0.00 B)

In [1]:
# Prepare tensors for training (Keras 3 string format)
train_sentences = tf.constant(train_df["text"].to_numpy(), dtype=tf.string)
train_labels = tf.constant(train_df["target"].to_numpy(), dtype=tf.int32)

val_sentences = tf.constant(val_df["text"].to_numpy(), dtype=tf.string)
val_labels = tf.constant(val_df["target"].to_numpy(), dtype=tf.int32)

# Compile model
model_6.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

# Fit model
model_6_history = model_6.fit(
    train_sentences, train_labels, epochs=5, validation_data=(val_sentences, val_labels)
)

NameError: name 'tf' is not defined

In [ ]:
# Make predictions on the validation set
model_6_pred_probs = model_6.predict(val_sentences)

# Convert prediction probabilities to 0 or 1 binary classes
model_6_preds = tf.squeeze(tf.round(model_6_pred_probs))

# Calculate evaluation metrics (Accuracy, Precision, Recall, F1-score)
from helper_functions import calculate_results

model_6_results = calculate_results(
    y_true=val_df["target"].to_numpy(), y_pred=model_6_preds
)
model_6_results

## Comparing the Performance of Each Model

Let's combine all of our model evaluation results into a single DataFrame to compare performance across all experiments.


In [ ]:
# Combine all model results into a DataFrame
all_model_results = pd.DataFrame(
    {
        "0_baseline": baseline_results,
        "1_simple_dense": model_1_results,
        "2_lstm": model_2_results,
        "3_gru": model_3_results,
        "4_bidirectional_lstm": model_4_results,
        "5_conv1d": model_5_results,
        "6_tf_hub_use_encoder": model_6_results,
    }
)
all_model_results = all_model_results.transpose()
all_model_results

In [ ]:
# Reduce the accuracy to same scale as other metrics (0-1)
all_model_results["accuracy"] = all_model_results["accuracy"] / 100

# Plot and compare all of the model results
import matplotlib.pyplot as plt

all_model_results.plot(kind="bar", figsize=(10, 7)).legend(bbox_to_anchor=(1.0, 1.0))
plt.title("Model Comparison on Disaster Tweet Classification")
plt.xlabel("Models")
plt.ylabel("Score")
plt.show()

In [ ]:
# Sort models by F1-score to find the best performing model
all_model_results.sort_values(by="f1", ascending=False)["f1"].plot(
    kind="bar", figsize=(10, 7)
)
plt.title("Models Sorted by F1-Score")
plt.ylabel("F1-Score")
plt.show()

## Saving and Loading the Best Model

In modern Keras 3, the recommended model format is `.keras` (native Keras format).


In [ ]:
# Save TF Hub USE model in modern .keras format
model_6.save("model_6_USE.keras")

In [ ]:
# Load in a model with custom layer registered
loaded_model_6 = tf.keras.models.load_model(
    "model_6_USE.keras", custom_objects={"USEEmbeddingLayer": USEEmbeddingLayer}
)

# Evaluate loaded model to confirm performance
loaded_model_6_pred_probs = loaded_model_6.predict(val_sentences)
loaded_model_6_preds = tf.squeeze(tf.round(loaded_model_6_pred_probs))
loaded_model_6_results = calculate_results(
    y_true=val_df["target"].to_numpy(), y_pred=loaded_model_6_preds
)
loaded_model_6_results == model_6_results

## Making Predictions on the Test Dataset

Now let's use our best model (Model 6: Universal Sentence Encoder) to make predictions on the unseen test dataset and display 10 tweets alongside their predicted values.


In [ ]:
# Convert test text column into string tensor for prediction
test_sentences = tf.constant(test_df["text"].to_numpy(), dtype=tf.string)

# Make predictions on test set with Model 6
test_pred_probs = model_6.predict(test_sentences)
test_preds = tf.squeeze(tf.round(test_pred_probs))

In [ ]:
# Create a DataFrame to inspect test tweets with their predictions
test_results_df = test_df.copy()
test_results_df["pred_prob"] = test_pred_probs
test_results_df["prediction"] = test_preds.numpy().astype(int)
test_results_df["pred_label"] = test_results_df["prediction"].apply(
    lambda x: "Real Disaster (1)" if x == 1 else "Not a Real Disaster (0)"
)

# Display first 10 results showing tweet text followed by its predicted value
for i, row in test_results_df.head(10).iterrows():
    print(f"--- Tweet #{i+1} ---")
    print(f"Tweet: {row['text']}")
    print(f"Prediction: {row['pred_label']}")
    print(f"Confidence (Pred Prob): {row['pred_prob']:.4f}\n")

In [ ]:
# Display first 10 test predictions in tabular format
test_results_df.head(10)[["text", "pred_label", "pred_prob"]]